<a href="https://colab.research.google.com/github/solosolve-ai/solosolve-ai/blob/main/manim_video_creator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#installs

In [ ]:
# ✅ Stable Manim installation for Google Colab (~5-10 min total)
# Run this cell only once when you start or restart the notebook session.
print("Starting installation... This may take 5-10 minutes.")
!sudo apt-get update -y
!sudo apt-get install -y libcairo2-dev libpango1.0-dev ffmpeg texlive-latex-base texlive-fonts-recommended texlive-fonts-extra texlive-latex-extra
!pip install --upgrade pip setuptools wheel
!pip install manim==0.19.0 manimpango==0.5.0
print("Installation complete!")

#video

In [ ]:
%%manim -pql DataDelugeAndSourceScene

from manim import *
import random

# --- Helper data for the animations ---

COMPLAINT_SNIPPETS = [
    "Wrong size sent!", "Arrived with a hole...", "Doesn't look like the picture.",
    "Material feels cheap.", "Fell apart after one wash.", "Color is completely off.",
    "This is not what I ordered.", "Took a month to arrive.", "Came with a noticeable stain.",
    "Poor quality stitching.", "Way, way too small.", "The zipper broke immediately.",
    "Smelled weird out of the bag.", "Shrank to half its size.", "Not cotton like it said.",
    "Threads are already loose.", "Looks like a cheap knockoff.", "Customer service won't reply."
]

# --- SCENE 1: The Data Deluge & Source ---

class DataDelugeAndSourceScene(MovingCameraScene):
    """
    Shows a chaotic stream of text that resolves into a structured database schema,
    followed by a SQL query extracting messy data.
    """
    def construct(self):
        # 1. The Deluge: A chaotic vortex of text
        self.camera.frame.save_state()
        chaos_group = VGroup()
        for _ in range(200):
            text = Text(
                random.choice(COMPLAINT_SNIPPETS),
                font_size=random.uniform(12, 24),
                color=random_color(),
                opacity=random.uniform(0.5, 1.0)
            )
            text.move_to(np.array([
                random.uniform(-10, 10),
                random.uniform(-6, 6),
                random.uniform(-5, 5)
            ]))
            chaos_group.add(text)

        self.play(FadeIn(chaos_group, scale=0.5))
        self.play(
            chaos_group.animate.shift(LEFT * 2).set_opacity(0.5),
            run_time=3,
            rate_func=linear
        )
        self.play(
            self.camera.frame.animate.scale(1.5),
            chaos_group.animate.scale(1.5),
            run_time=2
        )
        self.wait(1)

        # 2. Transition from Chaos to Structure
        schema_group, target_mobjects = self.create_schema_diagram()

        # We will create the illusion of transformation
        self.play(FadeOut(chaos_group, run_time=1.5))
        self.play(
            Restore(self.camera.frame),
            Create(schema_group),
            run_time=2
        )
        self.wait(2)

        # 3. The Query
        review_table = schema_group[2] # The VGroup for the Review table
        query_code = Code(
            code="SELECT text, rating\nFROM reviews\nWHERE rating <= 2;",
            language="sql",
            font_size=24,
            style="monokai"
        ).next_to(schema_group, RIGHT, buff=0.5)

        self.play(Write(query_code))
        self.wait(1)

        # 4. Extraction of messy data
        text_field = review_table[1][4] # Accessing the 'text' field
        rating_field = review_table[1][3] # Accessing the 'rating' field

        highlight_text = SurroundingRectangle(text_field, color=YELLOW)
        highlight_rating = SurroundingRectangle(rating_field, color=YELLOW)

        self.play(Create(highlight_text), Create(highlight_rating))

        arrow = Arrow(review_table.get_right(), review_table.get_right() + RIGHT*2, buff=0.2)
        messy_data_text = Text("<b>shrit</b>, 1.0", font="monospace", font_size=24).next_to(arrow, RIGHT)

        self.play(GrowArrow(arrow))
        self.play(Write(messy_data_text))
        self.wait(3)

    def create_table_mobject(self, title, fields):
        """Helper function to create a table for the schema."""
        title_text = Text(title, font_size=24, weight=BOLD).set_color(BLACK)
        title_box = VGroup(
            Circle(radius=0.2, color=GREEN_E, fill_opacity=1),
            Text("C", font_size=20, color=BLACK).move_to(ORIGIN)
        ).next_to(title_text, LEFT, buff=0.2)

        header = VGroup(title_box, title_text).arrange(RIGHT, buff=0.2)

        field_texts = VGroup(*[Text(f, font_size=20, font="monospace").set_color(BLACK) for f in fields])
        field_texts.arrange(DOWN, buff=0.25, aligned_edge=LEFT)

        table_content = VGroup(header, field_texts).arrange(DOWN, buff=0.4, aligned_edge=LEFT).set_z_index(2)

        bg_rect = Rectangle(
            width=table_content.get_width() + 0.5,
            height=table_content.get_height() + 0.5,
            color=GREY_E,
            fill_opacity=0.9
        ).set_z_index(1)
        bg_rect.move_to(table_content.get_center())

        return VGroup(bg_rect, table_content)

    def create_schema_diagram(self):
        """Creates the full three-table schema diagram."""
        user_table = self.create_table_mobject("User", ["+ {key} user_id : VARCHAR"])
        product_table = self.create_table_mobject("Product", [
            "+ {key} parent_asin : VARCHAR",
            "+ product_title : VARCHAR",
            "+ average_rating : FLOAT",
            "+ price : DECIMAL"
        ])
        review_table = self.create_table_mobject("Review", [
            "+ {key} review_id : INTEGER",
            "+ {FK} user_id : VARCHAR",
            "+ {FK} parent_asin : VARCHAR",
            "+ rating : FLOAT",
            "+ text : VARCHAR",
            "+ timestamp : BIGINT"
        ])

        user_table.to_corner(UL, buff=1)
        product_table.to_corner(UR, buff=1)
        review_table.next_to(VGroup(user_table, product_table), DOWN, buff=1.5)

        # Connections
        conn1 = Line(user_table.get_bottom(), review_table.get_top())
        conn2 = Line(product_table.get_bottom(), review_table.get_top())
        conn1_label = Text("1..* writes", font_size=20).next_to(conn1, LEFT, buff=0.1)
        conn2_label = Text("1..* receives", font_size=20).next_to(conn2, RIGHT, buff=0.1)

        schema_group = VGroup(user_table, product_table, review_table, conn1, conn2, conn1_label, conn2_label)

        # This is for a potential transformation animation, though we use FadeIn for simplicity.
        target_mobjects = VGroup(*user_table[1], *product_table[1], *review_table[1])

        return schema_group, target_mobjects

Manim Community v0.19.0

[11/01/25 17:22:01] WARNING  It looks like the scene contains a lot of sub-mobjects. Caching is      ]8;id=212440;file:///usr/local/lib/python3.12/dist-packages/manim/utils/hashing.py\hashing.py]8;;\:]8;id=892693;file:///usr/local/lib/python3.12/dist-packages/manim/utils/hashing.py#160\160]8;;\
                             sometimes not suited to handle such large scenes, you might consider                  
                             disabling caching with --disable_caching to potentially speed up the                  
                             rendering process.                                                                    

                    WARNING  You can disable this warning by setting disable_caching_warning to True ]8;id=234200;file:///usr/local/lib/python3.12/dist-packages/manim/utils/hashing.py\hashing.py]8;;\:]8;id=375700;file:///usr/local/lib/python3.12/dist-packages/manim/utils/hashing.py#166\166]8;;\
                             in your config file.                                                                  

[11/01/25 17:22:39] INFO     Animation 0 : Partial movie file written in                   ]8;id=939143;file:///usr/local/lib/python3.12/dist-packages/manim/scene/scene_file_writer.py\scene_file_writer.py]8;;\:]8;id=497396;file:///usr/local/lib/python3.12/dist-packages/manim/scene/scene_file_writer.py#588\588]8;;\
                             '/content/media/videos/content/480p15/partial_movie_files/Dat                         
                             aDelugeAndSourceScene/2456898656_1505103368_223132457.mp4'                            

[11/01/25 17:22:50] WARNING  It looks like the scene contains a lot of sub-mobjects. Caching is      ]8;id=47228;file:///usr/local/lib/python3.12/dist-packages/manim/utils/hashing.py\hashing.py]8;;\:]8;id=210509;file:///usr/local/lib/python3.12/dist-packages/manim/utils/hashing.py#160\160]8;;\
                             sometimes not suited to handle such large scenes, you might consider                  
                             disabling caching with --disable_caching to potentially speed up the                  
                             rendering process.                                                                    

...

[11/01/25 17:26:02] INFO     Animation 3 : Partial movie file written in                   ]8;id=989539;file:///usr/local/lib/python3.12/dist-packages/manim/scene/scene_file_writer.py\scene_file_writer.py]8;;\:]8;id=752674;file:///usr/local/lib/python3.12/dist-packages/manim/scene/scene_file_writer.py#588\588]8;;\
                             '/content/media/videos/content/480p15/partial_movie_files/Dat                         
                             aDelugeAndSourceScene/764728754_548731573_241551810.mp4'                              

<string>:110: DeprecationWarning: This method is not guaranteed to stay around. Please prefer getting the attribute normally.
<string>:111: DeprecationWarning: This method is not guaranteed to stay around. Please prefer getting the attribute normally.


[11/01/25 17:26:09] WARNING  It looks like the scene contains a lot of sub-mobjects. Caching is      ]8;id=502194;file:///usr/local/lib/python3.12/dist-packages/manim/utils/hashing.py\hashing.py]8;;\:]8;id=274892;file:///usr/local/lib/python3.12/dist-packages/manim/utils/hashing.py#160\160]8;;\
                             sometimes not suited to handle such large scenes, you might consider                  
                             disabling caching with --disable_caching to potentially speed up the                  
                             rendering process.                                                                    

                    WARNING  You can disable this warning by setting disable_caching_warning to True ]8;id=384089;file:///usr/local/lib/python3.12/dist-packages/manim/utils/hashing.py\hashing.py]8;;\:]8;id=244236;file:///usr/local/lib/python3.12/dist-packages/manim/utils/hashing.py#166\166]8;;\
                             in your config file.                                                                  

Animation 4: FadeOut(VGroup of 200 submobjects):  74%|███████▍  | 17/23 [00:18<00:07,  1.18s/it]